## Lab 03 impact analysis
In this notebook, we will compute an impact analysis that are usually performed to support emergency response efforts. 

**Table of Contents**
<br>
This notebook covers the below sections: 

1. [Background](#s4-1)
2. [Nepal June 2021 Flood Event Case Study](#s4-2)
    * [Sentinel-1 Data](#s4-2.1)
    * [Load Data](#s4-2.2)
    * [Visualize Data](#s4-2.3)
    * [Impact Analysis](#s4-2.4)
3. [Near-Real-Time FloodAI Dashboard](#s4-3)
4. [Summary](#s4-4)

## Requirements:

1. Downloading the [UNOSAT_case_study](https://www.dropbox.com/scl/fi/8attj6m097kjf7h9lr4mh/UNOSAT_case_study.zip?rlkey=pnc7qq2jpdbmp08dvi62whw5h&st=kcqesh2s&dl=0) 
1. Go to the "Side Bar", Click on "Data"
1. Upload it as `UNOSAT-case-study` (note it might take a bit of time to load and also to be visible by the notebook)

<a name='s4-1'></a>
## Background ##
Nepal is a hazard-prone country. Recurring disasters have been monitored throughout the years, including severe floods in 2008, 2014, and 2017. In 2015, following the 7.8-magnitude earthquake, UNOSAT's Emergency Mapping service was activated to provide satellite-derived damage assessment in support of the humanitarian response efforts on the ground. Acting as coordinator for mapping products, UNOSAT integrated the results from various sources in a publicly available web map. Very innovative at the time, this online viewing tool gathered all the analysis results. It also showed images collected from the ground through the newly developed UN ASIGN app. With the crowdsourcing app, individuals are able to send geotagged content, including photos of damages, to validate the satellite imagery-based assessment. Over the years, UNOSAT has worked with different partners in Nepal, providing technical support and implementing different activities to strengthen capacities in using Geospatial Information Technology (GIT) and data for Disaster Risk Management (DRM).  

<a name='s4-2'></a>
## Nepal June 2021 Flood Event Case Study ##
In a more recent incident that occurred during June 2021, Nepal was hit by heavy rainfall. Flooding and landslides were reported in many parts of the country and affected thousands of families<sup>[1]</sup>. Later in August of the same year, rainfall intensified again for several consecutive days, resulting in rivers overflowing, landslides triggered across the hills, and widespread inundation in the southern plains. In the affected areas, many houses, informal settlements, and other buildings were destroyed, forcing people to seek refuge with relatives or in temporary shelters. Health and sanitation concerns arose as access to running water became limited, increasing the risk of water-borne disease transmission, and limiting the ability to adhere to preventive measures against COVID-19.

[1] https://reliefweb.int/disaster/fl-2021-000134-npl

To rapidly respond, UNOSAT Rapid Mapping Service deployed the AI flood detection dashboard (shown below). Flood impact statistics such as sqkm of flood and flood affected population by admin units were automatically generated and displayed on the dashboard. UNOSAT FloodAI was able to process all the available Sentinel-1 overlapping with the area of interested shared by the Information Management Unit of the United Nation Resident Coordination Office in Nepal during the 29 days of activation. We will demonstrate the use of the flood segmentation model on Sentinel-1 images captured around July 2nd, 2021, to analyze the impact of the flood. 

<a name='s4-2.1'></a>
### Sentinel-1 Data ###
Sentinel-1 images can be viewed freely from the Alaska Satellite Facility (ASF) Data Search Vertex. Below are the instructions: 

1. Open the [ASF Data Search Vertex](https://search.asf.alaska.edu/#/?searchType=List%20Search)
2. Input the image name: `S1A_IW_GRDH_1SDV_20210702T001952_20210702T002017_038591_048DBF_C0E0` under **"List of Scene Name"**. You can also use [this link](https://search.asf.alaska.edu/#/?searchType=List%20Search&searchList=S1A_IW_GRDH_1SDV_20210702T001952_20210702T002017_038591_048DBF_C0E0&resultsLoaded=true&granule=S1A_IW_GRDH_1SDV_20210702T001952_20210702T002017_038591_048DBF_C0E0-GRD_HD&zoom=7.200&center=84.451,25.003). 
 <img src= "https://www.dropbox.com/scl/fi/ve1o7itztaw4ap9bqawef/ASF_screen.png?rlkey=cwl2slyc9wrw2v76bbw8bn8tx&st=arsn3lsd&dl=1" >
3. Find the image name in the bottom left corner
4. Click **"Zoom to scene"**
 <img src="https://www.dropbox.com/scl/fi/4i7oewexzkdycdmnpb47y/ASF_click.png?rlkey=io2xpmnf02d13nl9ef96ocipw&st=j6z7099g&dl=1" >
**Note**: no ASF login required. 

In [ ]:
# DO NOT CHANGE THIS CELL
# import dependencies
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import pyplot
import cv2 as cv
import os
import numpy as np
import warnings 
import gc
import random
import rasterio
from tqdm import tqdm
import math
import torch
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

<a name='s4-2.2'></a>
### Load Data ###
The Sentinel-1 images have been downloaded, pre-processed using the [ESA SNAP toolbox](https://step.esa.int/main/toolboxes/snap/), and compressed. Below, we will load the two polarizations of Sentinel-1, namely VV and VH, and combine them into an RGB image. 

In [ ]:
# Sentinel-1 is a radar satellite that transmits microwave pulses and records the
# backscattered energy.  It operates in two polarization modes:
#   VV: vertical transmit, vertical receive — sensitive to surface roughness and
#       soil moisture.  Calm water reflects almost all energy *away* from the
#       satellite (specular reflection) → dark pixels.
#   VH: vertical transmit, horizontal receive — sensitive to vegetation structure
#       and volume scattering.  Also dark over calm water.
#
# Building a false-colour RGB composite (R=VV, G=VH, B=VV-VH) makes it easy to
# distinguish flooded areas (dark, bluish) from vegetated / built-up land
# (brighter, more green/yellow).

img_VV = cv.imread("/kaggle/input/unosat-case-study/UNOSAT_case_study/images/VV_band/S1A_20210702T001952_C0E0_compressed.tif", -1)
img_VH = cv.imread("/kaggle/input/unosat-case-study/UNOSAT_case_study/images/VH_band/S1A_20210702T001952_C0E0_VH_compressed.tif", -1)

# Compose the three channels of the false-colour image
red   = img_VV           # VV polarization
green = img_VH           # VH polarization
blue  = img_VV - img_VH  # difference highlights contrast between the two

fig, (axr, axg, axb) = plt.subplots(1, 3, figsize=(21, 7))
axr.imshow(red,   cmap='Reds');   axr.set_title('Red channel — VV polarization')
axg.imshow(green, cmap='Greens'); axg.set_title('Green channel — VH polarization')
axb.imshow(blue,  cmap='Blues');  axb.set_title('Blue channel — VV−VH polarization')
plt.show()

In [ ]:
# Pixels where the VV value is exactly 0 are no-data / outside the radar swath.
# Visualising this mask helps confirm that the image frame is zero-padded rather
# than containing real measurements, which is important because:
#   1. The tiling pads the image with zeros — these must not be confused with
#      water pixels (which are also low-backscatter but non-zero).
#   2. The AnalysisExtent mask will later exclude these areas from the analysis.

no_data_mask = img_VV == 0
plt.figure(figsize=(8, 6))
plt.imshow(no_data_mask, cmap='gray')
plt.title('No-data pixels (VV == 0): white = no-data, black = valid pixel')
plt.colorbar(label='1 = no-data, 0 = valid')
plt.show()
print(f"No-data pixels: {no_data_mask.sum():,} / {no_data_mask.size:,} "
      f"({100 * no_data_mask.mean():.1f}%)")

In [ ]:
# The AnalysisExtent is a binary raster mask that delimits the geographic area
# of interest: Nawalparasi West, Rupandehi, and Kapilbastu districts in Lumbini
# province.  Pixels outside this region are excluded from the flood impact
# statistics to avoid counting areas that were not under active monitoring.

AnalysisExtent_path = "/kaggle/input/unosat-case-study/UNOSAT_case_study/images/VV_band/S1A_20210702T001952_C0E0_AnalysisExtent.tif"
if AnalysisExtent_path is not None:
    AnalysisExtent = cv.imread(AnalysisExtent_path, -1)
    plt.figure(figsize=(8, 6))
    plt.imshow(AnalysisExtent, cmap='gray')
    plt.title('Analysis Extent mask (white = area of interest)')
    plt.show()

### Tiling

The image's size is (21408,31094) therefore we have to tile it to run inference. 

**1. Create a tiling that returns tiles of size (256,256):** 

In [ ]:
def create_dir(dir_path):
    """Create *dir_path* (and any missing parents) if it does not already exist."""
    os.makedirs(dir_path, exist_ok=True)

In [ ]:
def padding(image):
    """Pad a 2-D image so that both dimensions are exact multiples of 256.

    Only the bottom and right edges are padded with zeros, so the spatial
    coordinates of every existing pixel are preserved.  After tiling and
    inference the padded border is ignored automatically because the
    AnalysisExtent mask excludes those pixels.

    Args:
        image (np.ndarray): 2-D array to pad.

    Returns:
        np.ndarray: Zero-padded array whose shape is a multiple of (256, 256).
    """
    # How many rows / columns are needed to reach the next multiple of 256?
    pad_h = int(np.ceil(image.shape[0] / 256)) * 256 - image.shape[0]
    pad_w = int(np.ceil(image.shape[1] / 256)) * 256 - image.shape[1]

    # Pad only the trailing edges (bottom, right) so the image origin is unchanged
    image_pad = np.pad(image, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
    print("Padded image size:", image_pad.shape)
    return image_pad

In [ ]:
def tiling(image_path, label_path, AnalysisExtent_path,
           tile_img_path, tile_lab_path, save_name, tile_size, offset,
           tile_extent_path=None):
    """Tile a large image (and optional label / analysis-extent mask) into fixed-size patches.

    The image is first zero-padded so that its dimensions are exact multiples of
    *tile_size*, then sliced into non-overlapping (or overlapping, depending on
    *offset*) patches that are saved as lossless PNG files.

    Args:
        image_path (str): Path to the main image GeoTIFF.
        label_path (str | None): Path to the label GeoTIFF, or None.
        AnalysisExtent_path (str | None): Path to the analysis-extent mask, or None.
        tile_img_path (str): Output directory for image tiles (must end with '/').
        tile_lab_path (str | None): Output directory for label tiles, or None.
        save_name (str): Tile filename prefix (must end with '_').
        tile_size (tuple): (width, height) of each tile in pixels.
        offset (tuple): (x_step, y_step) stride between tile origins.
            Set equal to tile_size for non-overlapping tiles.
        tile_extent_path (str | None): Output directory for extent mask tiles, or None.
    """
    create_dir(tile_img_path)

    # Read and pad the main image
    image = cv.imread(image_path, -1)
    image = padding(image)

    if label_path is not None:
        label = cv.imread(label_path, -1)
        label = padding(label)
        assert image.shape == label.shape, "Image and label must have the same shape after padding."
        create_dir(tile_lab_path)

    if AnalysisExtent_path is not None:
        extent = cv.imread(AnalysisExtent_path, -1)
        extent = padding(extent)
        if tile_extent_path is not None:
            create_dir(tile_extent_path)

    print("Tiling: started...")
    count = 0
    n_rows = int(math.ceil(image.shape[0] / offset[1]))
    n_cols = int(math.ceil(image.shape[1] / offset[0]))

    for i in tqdm(range(n_rows)):
        for j in range(n_cols):
            # Compute the pixel slice for this tile (clamped to image boundaries)
            r0 = offset[1] * i
            r1 = min(r0 + tile_size[1], image.shape[0])
            c0 = offset[0] * j
            c1 = min(c0 + tile_size[0], image.shape[1])
            tile_id = save_name + str(i) + "_" + str(j) + ".png"

            # Save the image tile as lossless PNG
            cv.imwrite(tile_img_path + tile_id, image[r0:r1, c0:c1], [cv.IMWRITE_PNG_COMPRESSION, 0])

            # Save the corresponding label tile (only if labels were provided)
            if label_path is not None:
                cv.imwrite(tile_lab_path + tile_id, label[r0:r1, c0:c1], [cv.IMWRITE_PNG_COMPRESSION, 0])

            # Save the analysis-extent tile (only if extent and output dir are provided)
            if AnalysisExtent_path is not None and tile_extent_path is not None:
                cv.imwrite(tile_extent_path + tile_id, extent[r0:r1, c0:c1], [cv.IMWRITE_PNG_COMPRESSION, 0])

            count += 1

    print(f"Tiling complete. Total tiles saved: {count}")

In [ ]:
# ── Input paths ────────────────────────────────────────────────────────────────
# Both bands are read from the Kaggle input dataset (read-only).
img_VV_path = "/kaggle/input/unosat-case-study/UNOSAT_case_study/images/VV_band/S1A_20210702T001952_C0E0_compressed.tif"
img_VH_path = "/kaggle/input/unosat-case-study/UNOSAT_case_study/images/VH_band/S1A_20210702T001952_C0E0_VH_compressed.tif"

# No ground-truth flood labels are available for this scene, so label tiles are
# not produced.  The AnalysisExtent mask is loaded separately (cell above) and
# applied during the impact analysis rather than being tiled.
label_path          = None
AnalysisExtent_path = None

# ── Output directories ─────────────────────────────────────────────────────────
# Kaggle's /kaggle/working/ directory is writable during the session.
tile_VV_img_path = "/kaggle/working/tiles_VV/"  # VV tiles output dir
tile_VH_img_path = "/kaggle/working/tiles_VH/"  # VH tiles output dir
tile_lab_path    = None  # no label tiles

# ── Tiling parameters ──────────────────────────────────────────────────────────
save_name = 'tile_'     # prefix; the function appends _<row>_<col>.png
tile_size = (256, 256)  # each tile is 256 × 256 pixels (matches model input)
offset    = (256, 256)  # stride == tile_size → non-overlapping tiles

In [ ]:
tiling(img_VV_path, label_path, AnalysisExtent_path,tile_VV_img_path,tile_lab_path,save_name,tile_size,offset)

In [ ]:
tiling(img_VH_path, label_path, AnalysisExtent_path,tile_VH_img_path,tile_lab_path,save_name,tile_size,offset)

### Testing
Let's test the tiling went well. (In case something went wrong in the tiling, we provide both VV and VH tiles here: `/kaggle/input/UNOSAT-case-study/UNOSAT_case_study/tiles_VV`  and `/kaggle/input/UNOSAT-case-study/UNOSAT_case_study/tiles_VH`).

In [ ]:
# List all tile filenames and verify that VV and VH produced the same number of tiles
tiles_VV = sorted(os.listdir(tile_VV_img_path))
tiles_VH = sorted(os.listdir(tile_VH_img_path))

print("Number of VV tiles:", len(tiles_VV))
print("Number of VH tiles:", len(tiles_VH))

# Sanity check: every VV tile must have a matching VH tile (same filename)
assert len(tiles_VV) == len(tiles_VH), "Mismatch between VV and VH tile counts!"
assert tiles_VV == tiles_VH, "VV and VH tile names do not match!"

### Visualize Data ###
The Sentinel-1 tiles are divided into tiles of size 256 x 256 so that they are easier to manipulate (as opposed to one large file). 

In [ ]:
# DO NOT CHANGE THIS CELL 
os.environ['NEPAL_VV_PATH']="/kaggle/working/tiles_VV"
os.environ['NEPAL_VH_PATH']="/kaggle/working/tiles_VH" 
 

In [ ]:
# DO NOT CHANGE THIS CELL
# define function to create RGB composite from VV and VH bands
def RGB_composite(tile_name): 
    """Function to create RGB composite from VV and VH bands"
    Args:
        tile_name: name of the tile
    Returns:
        stacked: RGB composite image
    """
    vv=mpimg.imread(os.path.join(os.getenv('NEPAL_VV_PATH'), tile_name))
    red=np.clip(vv, 0, 1)
    
    vh=mpimg.imread(os.path.join(os.getenv('NEPAL_VH_PATH'), tile_name))
    green=np.clip(vh, 0, 1)

    blue=np.clip((vv-vh), 0, 1)
    stacked=np.dstack((red, green, blue))
    return stacked

### Display one tile

In [ ]:
# DO NOT CHANGE THIS CELL
# display the RGB composite of one tile
import imageio
tile_name=random.choice(os.listdir(os.getenv('NEPAL_VV_PATH')))
print(f'Displaying {tile_name}.')

# create RGB composite
RGB_tile=RGB_composite(tile_name)
cmaps=['Reds', 'Greens', 'Blues']

# plot RGB bands
fig, ax=plt.subplots(1, 3, figsize=(15, 5))

for idx in range(3): 
    ax[idx].imshow(RGB_tile[:, :, idx], cmap=cmaps[idx])
    ax[idx].get_yaxis().set_visible(False)
    ax[idx].get_xaxis().set_visible(False)
    ax[idx].set_title(cmaps[idx])

print(f'The shape is {RGB_tile.shape}.')
plt.show()

A good exercise is to reload all the tiles to check if the tiling went well. 

**2. Reload the VV tiles and tile them back together**

In [ ]:
# DO NOT CHANGE THIS CELL
# plot composite for all tiles
fig=plt.figure(figsize=(10, 10))
ax=plt.subplot(111)
max_x, max_y=0, 0

for each_tile in os.listdir(os.environ['NEPAL_VH_PATH']): 
    if (each_tile.split('.')[-1]=='png'): 
        tile_name=each_tile.replace('.png', '')
        _, y, x=tile_name.split('_')
        y=int(y)
        x=int(x)
        if y>max_y: 
            max_y=y
        if x>max_x: 
            max_x=x
        image=mpimg.imread(os.path.join(os.environ['NEPAL_VH_PATH'], each_tile))
        plt.imshow(image, extent=(256*x, 256*(x+1), -256*(y+1), -256*y), cmap='gray')

ax.set_xlim([0, max_x*256])
ax.set_ylim([-max_y*256, 0])
ax.set_facecolor("black")
plt.show()

You can compare the composite with one that is displayed on ASF:

 <img src="https://www.dropbox.com/scl/fi/l186e14lm4d5cwvwbidjb/ASF_adjust.png?rlkey=ccu2usbgb0vgwp060rutpqai4&st=u2ncrijz&dl=1" >

Processing large image files is memory intensive. To ensure we have sufficient memory for the rest of the notebook, we free up some memory with `del()` and `gc.collect()`. 

In [ ]:
# DO NOT CHANGE THIS CELL
del(fig)
del(ax)
gc.collect()
!free

### Impact Analysis ###
When performing an impact analysis, we are concerned with:   
- What is the extent of the flood?
- How many people are likely affected or located within flood affected areas?
- What is the estimated number of critical facilities located within the flood affected areas? 

We will use baseline data, which refers to information that provides humanitarian actors with a pre-disaster picture of the potentially affected populations, critical infrastructure, basic geography, and other issues that will become crucial when disaster strikes. Baseline data are obtained from several different sources, each responsible for ensuring accuracy and integrity. Some examples of commonly used baseline data are:

- [Humanitarian Data Exchange](https://data.humdata.org/)
- [Common Operational Datasets ](https://cod.unocha.org/)
- [WorldPop](https://www.worldpop.org/)
- [OpenStreetMap](https://www.openstreetmap.org)
- [Esri Sentinel-2 10m Land Use/Land Cover Time Series](https://www.arcgis.com/home/item.html?id=cfcb7609de5f478eb7666240902d4d3d)

To estimate the population potentially affected by the flood, we will subtract the permanent water from the AI-based water extent. This will be the flood extent that didn't have water pre-disaster. We then intersect the flood extent with the open-source spatial demographic dataset [WorldPop](https://www.worldpop.org/). It is important to notice that there is a discrepancy between Sentinel-1 10-meter resolution and the 100-meter resolution of WorldPop. Resampling the population layer is required to align it with the flood extend before computing any impact analysis. The area of interest for the impact analysis includes the Nawalparasi West, Rupandehi and Kapilbastu districts, and Lumbini province. We curated the permanent water layer and the WorldPop dataset to easily compute the impact analysis. 


 <img src="https://www.dropbox.com/scl/fi/uo7djoogvm07ws4q59kmd/layers.png?rlkey=d5tph7mgww8svi93h56p7hu93&st=087wbila&dl=1" >

In [ ]:
# DO NOT CHANGE THIS CELL
# load a prediction obtained with a trained model to garantee equal results for all students.  
prediction = np.loadtxt("/kaggle/input/unosat-case-study/UNOSAT_case_study/gt/prediction.txt", dtype=int)
from matplotlib import pyplot
fig=plt.figure(figsize=(10, 10))
pyplot.imshow(prediction)
pyplot.show() 

#### How the flood impact is computed

The cell below follows four steps:

1. **Start from the AI flood prediction** — a binary raster where `1` = water detected by the model.
2. **Clip to the Area of Interest** — pixels outside `analysis_extent` are zeroed out.  This focuses the statistics on the monitored districts and avoids double-counting areas outside the UNOSAT mandate.
3. **Subtract permanent water** — rivers, lakes, and reservoirs that existed *before* the flood are removed using the `permanent_water` layer.  The remaining pixels represent *new* flood water, i.e., the true disaster impact.
4. **Estimate affected population** — the flood extent is intersected with the [WorldPop](https://www.worldpop.org/) 100 m population grid.  Because WorldPop encodes population counts in pixel values that have been rescaled to the `[0, 1]` float range of a PNG, they must be multiplied back by `255` (to undo the PNG normalisation) and then by `1/98.5066…` (to rescale WorldPop resolution to the Sentinel-1 resolution), yielding inhabitants per Sentinel-1 pixel.  Summing over all flooded pixels gives the total potentially affected population.

In [ ]:
# DO NOT CHANGE THIS CELL
# load baseline data
analysis_extent=mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/analysis_extent.png')
permanent_water=mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/permanent_water.png')
world_pop=mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/world_pop.png')

all_maps=[{'name': 'Area of Interest', 'map': mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/analysis_extent_sample.PNG')}, 
          {'name': 'Permanent Water Region', 'map': mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/permanent_water_sample.PNG')}, 
          {'name': 'Population', 'map': mpimg.imread('/kaggle/input/unosat-case-study/UNOSAT_case_study/baseline_maps/world_pop_sample.PNG')}]

# plot baseline data
fig, ax=plt.subplots(1, 3, figsize=(30, 10))

for idx, map in enumerate(all_maps): 
    ax[idx].imshow(map['map'])
    ax[idx].get_yaxis().set_visible(False)
    ax[idx].get_xaxis().set_visible(False)
    ax[idx].set_title(map['name'])

In [ ]:
# DO NOT CHANGE THIS CELL
# make a copy of the water extent
flood_extent=prediction.copy()

# focus on the region inside the analyis extent only
flood_extent[analysis_extent==0]=0 

# subtract permanent water 
flood_extent[permanent_water!=0]=0 

# calculate the potentially affected population resempling the WorldPop resolution to the Sentinel-1 resolution. 
affected_population=np.sum(world_pop*255*np.float32(flood_extent)/98.5066598616484) 

print(f"Population potentially affected: {affected_population}.") 

**3.The United Nations Satellite Centre (UNOSAT) map, displayed below, indicated a potential exposed population of 195'000  on June 2nd. Is it in line with the number you just obtained?**


 <img src="https://www.dropbox.com/scl/fi/h0bqxlneefr5uzdep9l1b/UNOSAT_A3_Natural_Landscape_FL20210630NPL_Lumbini_07072021-1.jpg?rlkey=vz5umezuym47a7v3avmhpi5my&st=pg4fsa4d&dl=1" >

<a name='s4-3'></a>
## Near-Real-Time FloodAI Dashboard ##
To run flood detection at scale and respond to a growing need for daily updates, it is crucial to develop end-to-end pipelines where satellite images are automatically downloaded and processed by machine learning algorithms to output flood extent. Tools that provide access and disseminate this information could play a key role, such as the [operational dashboards](https://unosat-geodrr.cern.ch/portal/apps/opsdashboard/index.html#/9ae9eea3c6054d36a5a128602ac6d9ac) that UNOSAT developed for the UN Regional Coordination Office in Nepal.

To ensure timely and accurate updates, the AI-generated flood extents are reviewed and corrected as necessary by the UNOSAT Rapid Mapping team, before feeding the dashboard for end user consumption. Another tailor-made feature was the integration of [UN ASIGN](https://www.unitar.org/sustainable-development-goals/united-nations-satellite-centre-unosat/our-portfolio/un-asign), an application that was shared with field officers across the country to upload geotagged information and photos in real time. The UN ASIGN application provided a way to collect and centralize ground data, which was extremely useful during the national lockdown. The visualization of all the data on the same platform provides a comprehensive overview of the area of interest despite movement restrictions.


 <img src="https://www.dropbox.com/scl/fi/x9k943antz6wujog6krbe/Nepal_dashboard_2021.png?rlkey=az8sh2poawsoy4a94lm0zicl8&st=f7b05dvh&dl=1" >


_This training material is property of the United Nations Satellite Centre (UNOSAT)- UNITAR and was designed, prepared, and delivered exclusively for this course. Please do not reproduce or distribute it without the expressed written permission of UNITAR-UNOSAT._

 <img src="https://www.dropbox.com/scl/fi/usm2smi9pn30f1e8xak6g/combined_logo.png?rlkey=pbsov0iblmupkp8eoo0zb8810&st=0ue11wh4&dl=1" >